# Скорость и перплексия Llama-3.2-1B-Instruct на WikiText-2

Исходная модель использует BF16. Во втором варианте веса всех 113 линейных слоев, включая `lm_head`, квантуются в INT4 с отдельным масштабом для каждой строки. Два веса занимают один байт.

Для запуска нужен GPU с поддержкой CUDA. Каталог эксперимента задается переменной `PROJECT` в первой ячейке. По умолчанию это папка `efficient_models_eval` в текущем рабочем каталоге. В пустой каталог клонируется ветка `main`. При повторном запуске используется существующая копия проекта.

In [1]:
import importlib.util
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("efficient_models_eval").expanduser().resolve()
REPO_URL = "https://github.com/EfficientNN/Efficient-Models-course-ITMO-2026.git"

PROJECT.mkdir(parents=True, exist_ok=True)
if not (PROJECT / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(PROJECT)],
        check=True,
    )

src_dir = PROJECT / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
revision = subprocess.check_output(
    ["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True
).strip()
print("Коммит:", revision)
print("Исходники:", src_dir)

Cloning into '/kaggle/working/efficient_models_eval'...


Коммит: aa60bc26e3127e6c4f198e638567c5e9f0d5a7d0
Исходники: /kaggle/working/efficient_models_eval/src


In [2]:
for package in ("transformers", "accelerate", "datasets", "triton"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import csv
import gc
import unittest

import torch
import triton
from IPython.display import FileLink, display

import test_evaluate_wikitext2
from evaluate_wikitext2 import EvalConfig, run_evaluation

assert torch.cuda.is_available(), "Для эксперимента требуется GPU с поддержкой CUDA"
torch.cuda.set_device(0)
for index in range(torch.cuda.device_count()):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")
print("Используется GPU 0")
print(
    "torch:",
    torch.__version__,
    "triton:",
    triton.__version__,
    "CUDA:",
    torch.version.cuda,
)
if torch.cuda.get_device_capability()[0] < 8:
    print(
        "Triton int4 matmul использует fp32 dot fallback. Совместимость проверяется ниже"
    )

GPU 0: Tesla T4
GPU 1: Tesla T4
Используется GPU 0
torch: 2.11.0+cu130 triton: 3.6.0 CUDA: 13.0
Triton int4 matmul использует fp32 dot fallback. Совместимость проверяется ниже


## Проверки

Перед загрузкой модели проверяются границы окон и расчет потерь. Затем на небольших матрицах результаты ядер квантизации и умножения сверяются с эталоном.

In [3]:
suite = unittest.defaultTestLoader.loadTestsFromModule(test_evaluate_wikitext2)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful()

spec = importlib.util.spec_from_file_location("kernel_checks", src_dir / "tests.py")
kernel_checks = importlib.util.module_from_spec(spec)
spec.loader.exec_module(kernel_checks)
torch.manual_seed(0)
with torch.inference_mode():
    print(kernel_checks.test_quant_kernel_equals_torch((17, 33), 8, True))
    print(
        "Matmul max_abs_err:",
        kernel_checks.test_matmul_matches_reference(17, 33, 65, 8, True),
    )
torch.cuda.synchronize()
print("Проверки ядер пройдены")

test_chunked_ce_matches_manual_shift (test_evaluate_wikitext2.LossTests.test_chunked_ce_matches_manual_shift) ... ok
test_last_logit_is_excluded_with_partial_chunk (test_evaluate_wikitext2.LossTests.test_last_logit_is_excluded_with_partial_chunk) ... ok
test_uniform_toy_model (test_evaluate_wikitext2.LossTests.test_uniform_toy_model) ... ok
test_every_target_once (test_evaluate_wikitext2.WindowTests.test_every_target_once) ... ok
test_exact_boundary_and_tail (test_evaluate_wikitext2.WindowTests.test_exact_boundary_and_tail) ... ok
test_invalid_inputs (test_evaluate_wikitext2.WindowTests.test_invalid_inputs) ... ok
test_partial_results (test_evaluate_wikitext2.WindowTests.test_partial_results) ... ok
test_results_and_ratios (test_evaluate_wikitext2.WindowTests.test_results_and_ratios) ... ok

----------------------------------------------------------------------
Ran 8 tests in 0.440s

OK


17x33
Matmul max_abs_err: 0.006558418273925781
Проверки ядер пройдены


## Проверочный запуск

На первых 8192 токенах выполняется весь расчет. Для каждой длины входа трижды измеряется время прямого прохода. Результаты сохраняются в `results/smoke`.

In [4]:
smoke = run_evaluation(
    EvalConfig(max_tokens=8192, repeats=3, output=str(PROJECT / "results/smoke"))
)
assert smoke["status"] == "complete"
assert smoke["bf16"]["perplexity"]["targets"] == smoke["int4"]["perplexity"]["targets"]
assert smoke["int4"]["quantized_linear"] == 113
gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

... (12 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


Оценка bf16, Linear int4: 0
Forward 128: 127.532 мс, 997.9 токенов/с
Forward 512: 507.291 мс, 1009.8 токенов/с
Forward 2048: 2436.995 мс, 839.6 токенов/с
PPL: окно 1/13, целей 2047
PPL: окно 11/13, целей 7167
PPL: окно 13/13, целей 8191
bf16: PPL=13.5085

Оценка int4, Linear int4: 113
Forward 128: 230.700 мс, 554.7 токенов/с
Forward 512: 896.299 мс, 570.8 токенов/с
Forward 2048: 4087.192 мс, 502.4 токенов/с
PPL: окно 1/13, целей 2047
PPL: окно 11/13, целей 7167
PPL: окно 13/13, целей 8191
int4: PPL=37.8457
Результаты: /kaggle/working/efficient_models_eval/results/smoke


## Оценка на всей тестовой выборке

Сначала оценивается модель в BF16, затем в INT4. Перплексия рассчитывается по одному окну за раз. Длина окна составляет 2048 токенов, шаг - 512.

Скорость измеряется отдельно для входов из 128, 512 и 2048 токенов. Для каждой длины после трех прогревов измеряется время десяти прямых проходов. KV-кеш отключен.

Результаты сохраняются в `results/full`.

In [5]:
full = run_evaluation(EvalConfig(output=str(PROJECT / "results/full")))
assert full["status"] == "complete"
print("PPL bf16:", full["bf16"]["perplexity"]["ppl"])
print("PPL int4:", full["int4"]["perplexity"]["ppl"])

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (289076 > 131072). Running this sequence through the model will result in indexing errors


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]


Оценка bf16, Linear int4: 0
Forward 128: 141.528 мс, 901.9 токенов/с
Forward 512: 558.936 мс, 915.7 токенов/с
Forward 2048: 2810.552 мс, 729.3 токенов/с
PPL: окно 1/562, целей 2047
PPL: окно 11/562, целей 7167
PPL: окно 21/562, целей 12287
PPL: окно 31/562, целей 17407
PPL: окно 41/562, целей 22527
PPL: окно 51/562, целей 27647
PPL: окно 61/562, целей 32767
PPL: окно 71/562, целей 37887
PPL: окно 81/562, целей 43007
PPL: окно 91/562, целей 48127
PPL: окно 101/562, целей 53247
PPL: окно 111/562, целей 58367
PPL: окно 121/562, целей 63487
PPL: окно 131/562, целей 68607
PPL: окно 141/562, целей 73727
PPL: окно 151/562, целей 78847
PPL: окно 161/562, целей 83967
PPL: окно 171/562, целей 89087
PPL: окно 181/562, целей 94207
PPL: окно 191/562, целей 99327
PPL: окно 201/562, целей 104447
PPL: окно 211/562, целей 109567
PPL: окно 221/562, целей 114687
PPL: окно 231/562, целей 119807
PPL: окно 241/562, целей 124927
PPL: окно 251/562, целей 130047
PPL: окно 261/562, целей 135167
PPL: окно 271/5

In [6]:
with (PROJECT / "results/full/summary.csv").open(
    encoding="utf-8", newline=""
) as stream:
    for row in csv.DictReader(stream):
        print(row)

{'variant': 'bf16', 'ppl': '11.493307320326538', 'ppl_delta': '0.0', 'ppl_relative_change': '0.0', 'length': '128', 'median_ms': '141.52823199998465', 'tokens_per_second': '901.9133646711898', 'speedup_bf16_over_int4': ''}
{'variant': 'bf16', 'ppl': '11.493307320326538', 'ppl_delta': '0.0', 'ppl_relative_change': '0.0', 'length': '512', 'median_ms': '558.935847999976', 'tokens_per_second': '915.7240418747741', 'speedup_bf16_over_int4': ''}
{'variant': 'bf16', 'ppl': '11.493307320326538', 'ppl_delta': '0.0', 'ppl_relative_change': '0.0', 'length': '2048', 'median_ms': '2810.552145999992', 'tokens_per_second': '729.2520311272503', 'speedup_bf16_over_int4': ''}
{'variant': 'int4', 'ppl': '34.07748312076961', 'ppl_delta': '22.58417580044307', 'ppl_relative_change': '1.9649849404532782', 'length': '128', 'median_ms': '236.30260400011593', 'tokens_per_second': '542.0117017531242', 'speedup_bf16_over_int4': '0.5989279407175522'}
{'variant': 'int4', 'ppl': '34.07748312076961', 'ppl_delta': '22

## Экспорт

Отчеты JSON и CSV из каталога `results` упаковываются в `exports/evaluation_results.zip` внутри каталога эксперимента. Путь к архиву выводится под ячейкой.

In [7]:
archive_path = Path(
    shutil.make_archive(
        str(PROJECT / "exports" / "evaluation_results"), "zip", PROJECT / "results"
    )
)
if archive_path.is_relative_to(Path.cwd()):
    display(FileLink(str(archive_path.relative_to(Path.cwd()))))
print("Архив результатов:", archive_path)

/kaggle/working/evaluation_results.zip

Архив результатов: /kaggle/working/evaluation_results.zip


## Метрики

- `ppl` - перплексия, `exp(total_nll / targets)`. Чем она ниже, тем лучше модель предсказывает следующий токен.
- `median_ms` - медианное время одного прямого прохода в миллисекундах.
- `tokens_per_second` - число входных токенов за все замеры, деленное на их суммарную длительность в секундах.
- `speedup_bf16_over_int4` - отношение медианного времени BF16 к медианному времени INT4. При значении выше 1 быстрее INT4, ниже 1 - BF16.
- `parameter_buffer_bytes` - суммарный размер параметров и буферов модели в байтах.

Время оценки перплексии (`evaluation_seconds`) включает перенос окон на GPU и расчет потерь. Время прямого прохода измеряется отдельно (`median_ms`).

[Методика PPL](https://huggingface.co/docs/transformers/perplexity). [WikiText-2](https://huggingface.co/datasets/Salesforce/wikitext).
